# Figure 2 — Policy path visualization

This notebook reproduces Figure 2 from Section 9.1. It estimates the full shift-policy path δ ↦ θ(δ) using Data-SMR, overlays the population Monte Carlo truth, and overlays the local approximation 2δθ_AME.

In [ ]:

from pathlib import Path
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path.cwd()
for parent in [REPO, *REPO.parents]:
    if (parent / "src" / "genriesz" / "scorematchingriesz.py").exists():
        REPO = parent
        break
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import genriesz.scorematchingriesz as smr
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
print("device:", DEVICE)


In [ ]:
N = 1000
N_FOLDS = 2
DELTAS = np.linspace(0.0, 1.5, 13)
N_MC_TRUTH = 200000
HIDDEN_DIMS = (256, 256, 256)
OUTCOME_EPOCHS = 200
SCORE_STEPS = 4000
BATCH_SIZE = 256
INTEGRATION_STEPS = 200
DSM_SIGMA_MIN = 0.01
DSM_SIGMA_MAX = 1.0
SIGMA_EVAL = 0.01
CLIP_LOG_RATIO = 20.0
TABLE_TITLE = "Figure 2 data: policy path estimates"
FIGURE_TITLE = "Figure 2: policy path visualization"

In [ ]:

def mu_function(x):
    x = np.asarray(x)
    return (
        1.0 + x[:, 0]
        + 0.1 * x[:, 0] ** 2
        + 2.0 * np.sin(x[:, 0])
        + x[:, 1]
        + x[:, 0] * x[:, 1]
        + x[:, 2] ** 2
        + x[:, 2] ** 3
    )


def partial_d_mu(x):
    x = np.asarray(x)
    return 1.0 + 0.2 * x[:, 0] + 2.0 * np.cos(x[:, 0]) + x[:, 1]


def sample_x(n, seed):
    rng = np.random.default_rng(seed)
    sigma = np.array([[1.0, 0.1, 0.1], [0.1, 1.0, 0.1], [0.1, 0.1, 1.0]])
    return rng.multivariate_normal(np.zeros(3), sigma, size=n).astype("float32")


def sample_observations(n, seed):
    rng = np.random.default_rng(seed)
    x = sample_x(n, seed)
    y = mu_function(x) + rng.normal(size=n)
    return x.astype("float32"), y.astype("float32")


def shift_x(x, delta):
    out = np.asarray(x, dtype="float32").copy()
    out[:, 0] += float(delta)
    return out


def true_ame(n_mc, seed=999):
    x = sample_x(n_mc, seed)
    return float(np.mean(partial_d_mu(x)))


def true_ape(delta, n_mc, seed=777):
    x = sample_x(n_mc, seed)
    return float(np.mean(mu_function(shift_x(x, delta)) - mu_function(shift_x(x, -delta))))


In [ ]:

x, y = sample_observations(N, RANDOM_SEED)
true_path = np.array([true_ape(delta, N_MC_TRUTH, seed=100 + i) for i, delta in enumerate(DELTAS)])
true_ame_value = true_ame(N_MC_TRUTH)

fold_score_values = {float(delta): np.zeros(N) for delta in DELTAS}
for train_idx, test_idx in smr.crossfit_splits(N, n_folds=N_FOLDS, seed=RANDOM_SEED):
    x_train, y_train = x[train_idx], y[train_idx]
    x_test, y_test = x[test_idx], y[test_idx]
    outcome = smr.fit_outcome_net(x_train, y_train, hidden_dims=HIDDEN_DIMS, n_epochs=OUTCOME_EPOCHS, batch_size=BATCH_SIZE, seed=RANDOM_SEED, device=DEVICE)
    gamma_hat = smr.predict_outcome(outcome, x_test, device=DEVICE).reshape(-1)
    residual = y_test - gamma_hat
    score_model = smr.fit_data_smr_score_dsm(x_train, hidden_dims=HIDDEN_DIMS, n_steps=SCORE_STEPS, batch_size=BATCH_SIZE, sigma_min=DSM_SIGMA_MIN, sigma_max=DSM_SIGMA_MAX, seed=RANDOM_SEED, device=DEVICE)
    for delta in DELTAS:
        m_gamma = smr.predict_outcome(outcome, shift_x(x_test, delta), device=DEVICE).reshape(-1) - smr.predict_outcome(outcome, shift_x(x_test, -delta), device=DEVICE).reshape(-1)
        if abs(float(delta)) < 1e-12:
            alpha_hat = np.zeros_like(residual)
        else:
            log_plus = smr.log_ratio_from_data_score_shift(score_model, x_test, float(delta), steps=INTEGRATION_STEPS, sigma_eval=SIGMA_EVAL, direction="+", normalize=True, x_p_for_norm=x_train, device=DEVICE).reshape(-1)
            log_minus = smr.log_ratio_from_data_score_shift(score_model, x_test, float(delta), steps=INTEGRATION_STEPS, sigma_eval=SIGMA_EVAL, direction="-", normalize=True, x_p_for_norm=x_train, device=DEVICE).reshape(-1)
            alpha_hat = np.exp(np.clip(log_plus, -CLIP_LOG_RATIO, CLIP_LOG_RATIO)) - np.exp(np.clip(log_minus, -CLIP_LOG_RATIO, CLIP_LOG_RATIO))
        fold_score_values[float(delta)][test_idx] = m_gamma + alpha_hat * residual

rows = []
for delta in DELTAS:
    est = smr.wald_interval(fold_score_values[float(delta)])
    rows.append({"delta": float(delta), "estimate": est.estimate, "se": est.se, "ci_low": est.ci_low, "ci_high": est.ci_high})
path_results = pd.DataFrame(rows)
path_results["truth"] = true_path
path_results["local_linear"] = 2.0 * path_results["delta"] * true_ame_value
print(TABLE_TITLE)
display(path_results)


In [ ]:

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(path_results["delta"], path_results["estimate"], label="estimated path from AME score")
ax.fill_between(path_results["delta"], path_results["ci_low"], path_results["ci_high"], alpha=0.2, label="95% CI")
ax.plot(path_results["delta"], path_results["truth"], linestyle="--", label="true path")
ax.plot(path_results["delta"], path_results["local_linear"], linestyle=":", label="2δ × true AME")
ax.set_title(FIGURE_TITLE)
ax.set_xlabel("policy shift δ")
ax.set_ylabel("policy effect θ(δ)")
ax.legend()
fig.tight_layout()
plt.show()
